# Map Stockholm POIs to stadsdelar (2021 boundaries)

Maps POI latitude/longitude to **stadsdel** (smaller geographic districts) using Stockholm City boundary data archived in [krasch/stockholm-stadsdelar](https://github.com/krasch/stockholm-stadsdelar). The polygons reflect **2021**, not the current administrative arrangement. This boundary GeoJSON contains a `name` attribute for each smaller district, but **does not include a larger administrative-area field**, so none is invented here.

The boundary file excludes water areas. Any points in water or outside the boundaries remain unassigned. Run all cells with internet access the first time to download the boundary file, or manually place the linked GeoJSON beside this notebook.

In [1]:
# Run only if dependencies are missing:
# %pip install geopandas pandas requests shapely pyproj

In [2]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import requests

INPUT = Path("stockholm_poi_enriched_analysis.csv")
OUTPUT = Path("stockholm_poi_with_districts.csv")
BOUNDARIES = Path("stockholm_stadsdelar_2021.geojson")
URL = ("https://raw.githubusercontent.com/krasch/stockholm-stadsdelar/"
       "master/geojson/simplified/stadsdelar_utan_vatten.geo.json")

if not BOUNDARIES.exists():
    response = requests.get(URL, timeout=90)
    response.raise_for_status()
    BOUNDARIES.write_bytes(response.content)

pois = pd.read_csv(INPUT, low_memory=False)
polygons = gpd.read_file(BOUNDARIES).to_crs("EPSG:4326")
print("Boundary fields:", polygons.columns.tolist())
assert "name" in polygons.columns, "Expected district names under the 'name' field."
polygons = polygons[["name", "geometry"]].rename(columns={"name": "district"})
polygons["district"] = polygons["district"].astype("string").str.strip().str.title()
print(f"POIs: {len(pois):,} | district polygons: {len(polygons)}")

Boundary fields: ['name', 'geometry']
POIs: 4,234 | district polygons: 117


In [3]:
# Spatial join. 'intersects' includes points precisely on the boundary.
# Resetting a numeric row ID lets us safely restore the original POI order.
pois = pois.reset_index(drop=True)
pois["_poi_row"] = pois.index
points = gpd.GeoDataFrame(
    pois,
    geometry=gpd.points_from_xy(pois["longitude"], pois["latitude"]),
    crs="EPSG:4326"
)
joined = gpd.sjoin(points, polygons, how="left", predicate="intersects")

# If a point touches multiple polygons, keep one and flag it for inspection.
match_counts = joined.groupby("_poi_row")["district"].count()
ambiguous = (match_counts > 1).sum()
joined = joined.sort_values(["_poi_row", "district"]).drop_duplicates("_poi_row")

result = pd.DataFrame(joined.drop(columns=["geometry", "index_right"], errors="ignore"))
result = result.sort_values("_poi_row").drop(columns="_poi_row")
result["boundary_year"] = 2021
result["boundary_source"] = "Stockholm Stad / krasch stockholm-stadsdelar (2021)"
result["district_match_status"] = result["district"].notna().map({True:"matched", False:"unmatched"})

assert len(result) == len(pois), "Rows changed unexpectedly after spatial join"
result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"Saved {OUTPUT} with {len(result):,} rows")
print("Matched:", result["district"].notna().sum())
print("Unmatched:", result["district"].isna().sum())
print("Points touching multiple polygons:", ambiguous)
print("\nTop districts:")
print(result["district"].value_counts().head(15))
display(result[["name", "analysis_group", "district", "district_match_status"]].head(12))

Saved stockholm_poi_with_districts.csv with 4,234 rows
Matched: 4148
Unmatched: 86
Points touching multiple polygons: 0

Top districts:
district
Södermalm               694
Norrmalm                460
Vasastaden              428
Östermalm               264
Kungsholmen             206
Djurgården              196
Gamla Stan              184
Norra Djurgården         97
Ladugårdsgärdet          85
Södra Hammarbyhamnen     81
Johanneshov              66
Liljeholmen              64
Kista                    59
Årsta                    50
Midsommarkransen         46
Name: count, dtype: Int64


,name,analysis_group,district,district_match_status
0,Stockholms stadshus,culture_heritage,Kungsholmen,matched
1,Stockholms slott,visitor_attraction,Gamla Stan,matched
2,Riksdagshuset,culture_heritage,Gamla Stan,matched
3,Historiska museet,visitor_attraction,Östermalm,matched
4,Arvfurstens palats,culture_heritage,Norrmalm,matched
5,Ålstensskogen,parks_recreation,Ålsten,matched
6,Ulvsunda slott,culture_heritage,Ulvsunda,matched
7,Ekensbergs Båtsällskap,parks_recreation,<NA>,unmatched
8,Bååtska palatset,culture_heritage,Norrmalm,matched
9,Hallwylska palatset,visitor_attraction,Norrmalm,matched
